In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
# from moc.models.mixture.mixture_model import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
from moc.metrics.distribution_metrics import pce
import torch
import pickle

In [ ]:
data = {
    "Datasets": [
        "households", "scm20d", "rf2", "rf1", "scm1d", "meps21", "meps19", "meps20",
        "house", "bio", "blog data", "calcofi", "taxi"
    ],
    "Marginal": [0.0525, 0.0331, 0.0523, 0.0569, 0.0280, 0.0607, 0.0721, 0.0627, 0.0446, 0.0565, 0.0675, 0.0401, 0.0418],
    "Loc.":     [0.0457, 0.0404, 0.0435, 0.0400, 0.0248, 0.0549, 0.0635, 0.0590, 0.0451, 0.0809, 0.0398, 0.0537, 0.0292],
    "Scale":    [0.0844, 0.0948, 0.1175, 0.1171, 0.0913, 0.0730, 0.0822, 0.0831, 0.0349, 0.0568, 0.0508, 0.0302, 0.0863],
    "Dep.":     [0.0249, 0.0857, 0.1409, 0.1128, 0.0618, 0.4900, 0.4900, 0.4900, 0.4900, 0.4900, 0.4900, 0.4900, 0.4900],
    "PCA":      [0.0359, 0.0412, 0.0465, 0.0573, 0.0303, 0.0528, 0.0603, 0.0601, 0.0440, 0.0548, 0.0372, 0.0326, 0.0459],
    "HDR":      [0.0696, 0.1417, 0.3360, 0.3061, 0.1478, 0.1440, 0.1836, 0.1238, 0.0335, 0.2129, 0.0621, 0.0714, 0.1192]
}

df = pd.DataFrame(data)

df.to_csv("real_experiments_gaussian_nll.csv", index=False)

In [ ]:
config = get_config()
config.device = 'cpu'

def compute_null_hyp(nb_test_samples=1000, n_runs=10, n_points=100):
    pits = np.random.rand(nb_test_samples, n_runs, n_points)
    pits_tensor = torch.tensor(pits, dtype=torch.float32)
    sorted_pit = torch.sort(pits_tensor, dim=-1)[0]
    n = pits.shape[-1]
    lin = (torch.arange(n) + 1) / n
    calib = (sorted_pit - lin).abs().pow(1).mean(dim=-1).numpy()
    assert calib.ndim == 2
    return calib.mean(axis=-1)

def cal_PCE(nb_test_samples=1000, n_runs=10, n_points=100):
    pits = np.random.rand(nb_test_samples, n_runs, n_points)
    pits_tensor = torch.tensor(pits, dtype=torch.float32)
    sorted_pit = torch.sort(pits_tensor, dim=-1)[0]
    n = pits.shape[-1]
    lin = (torch.arange(n) + 1) / n
    calib = (sorted_pit - lin).abs().pow(1).mean(dim=-1).numpy()
    return calib

def plot_pce_comparison(pces_real, test_statistics):

    dataset_names = list(pces_real.keys())
    real_pces = [pces_real[name] for name in dataset_names]
    null_means = [test_statistics[name].mean() for name in dataset_names]
    null_stds = [test_statistics[name].std() for name in dataset_names]

    x = np.arange(len(dataset_names))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width/2, null_means, width, label='Null hypothesis (mean PCE)', yerr=null_stds, capsize=5)
    plt.bar(x + width/2, real_pces, width, label='Real model PCE')

    plt.xticks(x, dataset_names, rotation=45, ha='right')
    plt.ylabel("PCE")
    plt.title("PCE comparison: Null hypothesis vs Real model")
    plt.legend()
    plt.tight_layout()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
dataset_names = [
     #['camehl', 'households'],
     #             ['mulan', 'scm20d'],
     #             ['mulan', 'rf2'],
     #             ['mulan', 'rf1'],
     #             ['mulan', 'scm1d'],
     #             ['feldman', 'meps_21'],
     #             ['feldman', 'meps_19'],
     #             ['feldman', 'meps_20'],
     #             ['feldman', 'house'],
     #             ['feldman', 'bio'],
      #            ['feldman', 'blog_data'],
                 ['del_barrio', 'calcofi'],
                ['wang', 'taxi']
                 ]
pces_across_datasets = {}
test_statistics = {}
seed = 42   
nb_test_samples= 1000
n_runs = 100
for dataset in dataset_names:
    data_group, data_name = dataset
    print(f"Working on dataset {data_name}")
    rc = RunConfig(config, data_group, data_name)
    datamodule = RealDataModule(rc, seed=seed)
    n = datamodule.total_size
    n *= datamodule.train_val_calib_test_split_ratio[-1]
    n = int(n)
    null_hyp = compute_null_hyp(nb_test_samples, n_runs, n)
    test_statistics[data_name] = null_hyp